# Options from Zero · Episode 23: Changing the odds: martingales and Girsanov

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. The wrong way · 4. Martingales · 5. Reweighting the paths · 6. Same paths, right price · 7. Two probabilities · 8. This episode's question · 9. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
mu = 0.08   # the share's real-world expected return (illustrative)

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 23,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

In [ ]:
out["trade"] = {"spot": S, "strike": K, "years": T, "rate_pct": r * 100, "vol_pct": vol * 100}

Part 2's additions: Greeks, simulation, hedging, implied volatility, Black and Bachelier (`episodes/_shared/options_part2.py`):

In [ ]:
# Part 2 additions to options.py: the Greeks, simulation, a hedging simulator, implied volatility, and Black's and
# Bachelier's formulas on a forward (for FX and interest-rate options). Written out by hand and copied verbatim into
# each Part 2 notebook; the notebooks check them against QuantLib. Needs options.py (norm_cdf, black_scholes),
# math and numpy as np.

def norm_pdf(x):
    """The bell curve's height at x (standard normal density)."""
    return math.exp(-x * x / 2) / math.sqrt(2 * math.pi)

def norm_cdf_v(x):
    """norm_cdf for a whole numpy array at once."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x, dtype=float) / math.sqrt(2)))

def bs_greeks(s, k, t, r, vol):
    """Price and Greeks of a European call on a share with no dividends.
    delta per $1 of share price, gamma per $1 squared, vega per 1.00 of volatility (100 vol points),
    theta per year (negative: value lost as time passes), rho per 1.00 of interest rate."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    disc = math.exp(-r * t)
    return {"price": s * norm_cdf(d1) - k * disc * norm_cdf(d2),
            "delta": norm_cdf(d1),
            "gamma": norm_pdf(d1) / (s * vol * math.sqrt(t)),
            "vega": s * norm_pdf(d1) * math.sqrt(t),
            "theta": -s * norm_pdf(d1) * vol / (2 * math.sqrt(t)) - r * k * disc * norm_cdf(d2),
            "rho": k * t * disc * norm_cdf(d2)}

def call_delta(s, k, t, r, vol):
    """Black-Scholes delta N(d1) for an array of share prices."""
    d1 = (np.log(np.asarray(s, dtype=float) / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    return norm_cdf_v(d1)

def simulate_paths(s, t, drift, vol, steps, n_paths, rng):
    """Share price paths: shape (n_paths, steps + 1), starting at s, lognormal steps with the given drift."""
    dt = t / steps
    z = rng.standard_normal((n_paths, steps))
    log_steps = (drift - vol ** 2 / 2) * dt + vol * math.sqrt(dt) * z
    return s * np.exp(np.concatenate([np.zeros((n_paths, 1)), np.cumsum(log_steps, axis=1)], axis=1))

def monte_carlo_call(s, k, t, r, vol, n, rng):
    """Price = the average discounted payoff over n risk-neutral outcomes, with its standard error."""
    z = rng.standard_normal(n)
    s_t = s * np.exp((r - vol ** 2 / 2) * t + vol * math.sqrt(t) * z)
    pv = math.exp(-r * t) * np.maximum(s_t - k, 0.0)
    return float(pv.mean()), float(pv.std(ddof=1) / math.sqrt(n))

def hedge_short_call(s, k, t, r, vol_implied, vol_realised, steps, n_paths, rng, hedge=True):
    """Sell the call at vol_implied and, if hedge, rebalance to delta shares (computed at vol_implied) `steps` times.
    The share moves with vol_realised and grows at r. Cash earns r. Returns each path's P&L at expiry in today's dollars."""
    dt = t / steps
    price = s * np.ones(n_paths)
    held = call_delta(price, k, t, r, vol_implied) if hedge else np.zeros(n_paths)
    cash = black_scholes(s, k, t, r, vol_implied) - held * price      # premium received, shares bought
    for i in range(1, steps + 1):
        price = price * np.exp((r - vol_realised ** 2 / 2) * dt + vol_realised * math.sqrt(dt) * rng.standard_normal(n_paths))
        cash = cash * math.exp(r * dt)
        if hedge and i < steps:
            new = call_delta(price, k, t - i * dt, r, vol_implied)
            cash = cash - (new - held) * price                        # buy (or sell) the difference
            held = new
    pnl = cash + held * price - np.maximum(price - k, 0.0)           # sell the shares, pay the option's payoff
    return pnl * math.exp(-r * t)

def implied_vol(price, s, k, t, r, lo=0.0001, hi=3.0, tol=1e-12):
    """Run Black-Scholes backwards by bisection: halve [lo, hi] until the price fits. Returns (vol, halvings)."""
    n = 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if black_scholes(s, k, t, r, mid) < price:
            lo = mid
        else:
            hi = mid
        n += 1
    return (lo + hi) / 2, n

def black76(forward, strike, t, vol, df, call=True):
    """Black's formula on a forward: df x [F N(d1) - K N(d2)] for a call. FX options (Garman-Kohlhagen) and caplets."""
    sd = vol * math.sqrt(t)
    d1 = (math.log(forward / strike) + sd ** 2 / 2) / sd
    d2 = d1 - sd
    if call:
        return df * (forward * norm_cdf(d1) - strike * norm_cdf(d2))
    return df * (strike * norm_cdf(-d2) - forward * norm_cdf(-d1))

def bachelier(forward, strike, t, normal_vol, df, call=True):
    """Bachelier's formula: the forward moves by normal_vol x sqrt(t) in absolute terms (it can go below zero)."""
    sd = normal_vol * math.sqrt(t)
    d = (forward - strike) / sd
    if call:
        return df * ((forward - strike) * norm_cdf(d) + sd * norm_pdf(d))
    return df * ((strike - forward) * norm_cdf(-d) + sd * norm_pdf(d))

## 2. Last episode's question

A density integrates to 1: the total probability of all outcomes.

## 3. The wrong way

Simulate the share in the **real world**, growing at its expected return $\mu$ = 8%, average the call's payoffs, and discount at the risk-free rate. The answer is too high: the payoff is risky, and 3.60% is the rate for riskless money.

In [ ]:
out["answer"] = {"integral": 1}
n = 1_000_000
z = np.random.default_rng(23).standard_normal(n)        # the path's Brownian motion at T, divided by sqrt(T)
ST = S * np.exp((mu - vol ** 2 / 2) * T + vol * math.sqrt(T) * z)
pay = np.maximum(ST - K, 0.0)
formula = black_scholes(S, K, T, r, vol)
out["naive"] = {"price": float(math.exp(-r * T) * pay.mean()), "se": float(math.exp(-r * T) * pay.std() / math.sqrt(n)),
                "mu_pct": mu * 100, "r_pct": r * 100, "formula": formula, "paths": n}
out["naive"]

## 4. Martingales

Under the risk-neutral odds, any price discounted at the risk-free rate has no expected drift: its expected future value is today's value. A process like that is a **martingale**. Under the real-world odds our share's discounted price drifts up at $\mu - r$; under the risk-neutral odds it doesn't.

In [ ]:
disc_mean_P = float(math.exp(-r * T) * ST.mean())
out["martingale"] = {"discounted_mean_real": disc_mean_P, "drift_real_pct": (mu - r) * 100, "today": S}
out["martingale"]

## 5. Reweighting the paths

Girsanov's theorem (1960): changing the odds from real-world to risk-neutral is the same as giving each path a weight

$$L = \exp\Big(-\lambda W_T - \tfrac12\lambda^2 T\Big), \qquad \lambda = \frac{\mu - r}{\sigma}$$

where $W_T = \sqrt{T}\,Z$ is the path's Brownian motion. $\lambda$ is the **market price of risk**: the extra return per unit of volatility. The weights average to 1, and they shift the drift from $\mu$ to $r$ while leaving the volatility untouched. Paths that end high get weights below 1; paths that end low, above 1.

In [ ]:
lam = (mu - r) / vol
L = np.exp(-lam * math.sqrt(T) * z - 0.5 * lam ** 2 * T)
grid = np.linspace(-3, 3, 121)
s_grid = S * np.exp((mu - vol ** 2 / 2) * T + vol * math.sqrt(T) * grid)
w_grid = np.exp(-lam * math.sqrt(T) * grid - 0.5 * lam ** 2 * T)
out["weights"] = {"lambda": lam, "mean_L": float(L.mean()), "se_L": float(L.std() / math.sqrt(n)),
                  "reweighted_discounted_mean": float(math.exp(-r * T) * (L * ST).mean()),
                  "s": s_grid.round(3).tolist(), "w": w_grid.tolist(), "ones": [1.0] * len(grid),
                  "high": 150.0, "low": 80.0, "w_at_150": float(np.interp(150, s_grid, w_grid)), "w_at_80": float(np.interp(80, s_grid, w_grid)),
                  "ticks": [{"x": float(v), "label": f"${v}"} for v in (60, 80, 100, 120, 140, 160, 180)]}
assert abs(L.mean() - 1) < 3 * out["weights"]["se_L"] + 1e-3
out["weights"] = {k: v for k, v in out["weights"].items()}
{k: v for k, v in out["weights"].items() if not isinstance(v, list)}

## 6. Same paths, right price

Average the weighted payoffs of the *same* real-world paths, discount at the risk-free rate, and the price comes out right.

In [ ]:
wp = math.exp(-r * T) * L * pay
out["reweighted"] = {"price": float(wp.mean()), "se": float(wp.std() / math.sqrt(n)), "formula": formula}
out["reweighted"]["z"] = (out["reweighted"]["price"] - formula) / out["reweighted"]["se"]
assert abs(out["reweighted"]["z"]) < 3
out["reweighted"]

## 7. Two probabilities

The chance the call is exercised is different under the two sets of odds. Under the risk-neutral odds it is Part 1's $N(d_2)$.

In [ ]:
d2 = (math.log(S / K) + (r - vol ** 2 / 2) * T) / (vol * math.sqrt(T))
out["probs"] = {"real_world": float((ST > K).mean()), "risk_neutral": float((L * (ST > K)).mean()), "N_d2": norm_cdf(d2)}
out["table"] = [{"method": "Real-world odds, discounted at r", "price": out["naive"]["price"], "exercise": out["probs"]["real_world"]},
                {"method": "Same paths, reweighted", "price": out["reweighted"]["price"], "exercise": out["probs"]["risk_neutral"]},
                {"method": "Black-Scholes", "price": formula, "exercise": out["probs"]["N_d2"]}]
pd.DataFrame(out["table"])

## 8. This episode's question

If $\mu$ equalled $r$, the market price of risk $\lambda$ would be zero, so every weight would be exactly 1: the real-world and risk-neutral odds would be the same.

In [ ]:
out["question"] = {"lambda": 0.0, "weight": 1.0}
out["history"] = {"girsanov": 1960}

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")